# LeetCode 2304 — Minimum Path Cost in a Grid bằng Python

Link đề: https://leetcode.com/problems/minimum-path-cost-in-a-grid/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, Matrix

Skeleton LeetCode:

```python
class Solution:
    def minPathCost(self, grid: List[List[int]], moveCost: List[List[int]]) -> int:
        ...
```

## 1. Mục tiêu học

1. Mô hình hóa bài thành **đồ thị phân tầng** (layered graph): mỗi hàng là một tầng, cạnh chỉ đi từ tầng `i` sang tầng `i + 1`.
2. Viết DP theo hàng `f[i][j]` = chi phí nhỏ nhất của đường đi kết thúc ở ô `(i, j)`, và **nén** còn hai hàng.
3. Đọc kỹ cách đánh chỉ số chi phí: `moveCost[giá trị ô xuất phát][cột đích]` — **giá trị** ô, không phải chỉ số cột.
4. So sánh DP với vét cạn `nᵐ` đường đi, với đệ quy có nhớ, và với Dijkstra trên đồ thị `m·n` đỉnh.
5. Truy vết đường đi tối ưu và mở rộng sang các biến thể (đếm đường đi tối ưu, chỉ được đi sang cột kề…).

## 2. Đề bài & ví dụ

Cho ma trận `grid` `m × n` gồm các số nguyên **phân biệt** từ `0` đến `m·n − 1`. Từ ô `(x, y)` với `x < m − 1` có thể đi tới **bất kỳ** ô nào của hàng kế tiếp `(x + 1, 0), …, (x + 1, n − 1)`. Không thể đi tiếp từ hàng cuối.

Mỗi bước đi có chi phí cho bởi `moveCost` kích thước `(m·n) × n`: `moveCost[i][j]` là chi phí đi từ ô **có giá trị `i`** tới ô ở **cột `j`** của hàng kế tiếp.

Chi phí của một đường đi = tổng **giá trị các ô** đã đi qua + tổng **chi phí các bước đi**. Trả về chi phí nhỏ nhất của một đường đi bắt đầu ở một ô bất kỳ của hàng đầu và kết thúc ở một ô bất kỳ của hàng cuối.

**Ví dụ 1:** `grid = [[5,3],[4,0],[2,1]]`, `moveCost = [[9,8],[1,5],[10,12],[18,6],[2,4],[14,3]]` → `17` (đường `5 → 0 → 1`: ô `5 + 0 + 1 = 6`, bước `3 + 8 = 11`).

**Ví dụ 2:** `grid = [[5,1,2],[4,0,3]]`, `moveCost = [[12,10,15],[20,23,8],[21,7,1],[8,1,13],[9,10,25],[5,3,2]]` → `6` (đường `2 → 3`: `5 + 1`).

**Ràng buộc:** `2 ≤ m, n ≤ 50`; `1 ≤ moveCost[i][j] ≤ 100`.

In [1]:
from typing import List
from functools import lru_cache
from itertools import product
from math import inf
import heapq, random, time

examples = [([[5, 3], [4, 0], [2, 1]], [[9, 8], [1, 5], [10, 12], [18, 6], [2, 4], [14, 3]], 17),
            ([[5, 1, 2], [4, 0, 3]], [[12, 10, 15], [20, 23, 8], [21, 7, 1], [8, 1, 13], [9, 10, 25], [5, 3, 2]], 6)]
for g, mc, exp in examples:
    print("grid:", g, "-> kỳ vọng", exp)

grid: [[5, 3], [4, 0], [2, 1]] -> kỳ vọng 17
grid: [[5, 1, 2], [4, 0, 3]] -> kỳ vọng 6


## 3. Kiến thức nền

### 3.1 Chi phí một đường đi

Một đường đi là dãy cột `c₀, c₁, …, c_{m−1}` (một cột cho mỗi hàng). Chi phí:

`Σᵢ grid[i][cᵢ] + Σ_{i<m−1} moveCost[grid[i][cᵢ]][c_{i+1}]`.

In [2]:
def path_cost(grid, moveCost, cols):
    total = sum(grid[i][c] for i, c in enumerate(cols))
    total += sum(moveCost[grid[i][cols[i]]][cols[i + 1]] for i in range(len(cols) - 1))
    return total

g, mc, _ = examples[0]
print("đường 5 -> 0 -> 1 = cột", [0, 1, 1], "chi phí", path_cost(g, mc, [0, 1, 1]))

đường 5 -> 0 -> 1 = cột [0, 1, 1] chi phí 17


### 3.2 Chỉ số của `moveCost`: **giá trị** ô nguồn, **cột** ô đích

`moveCost` có `m·n` hàng — một hàng cho mỗi **giá trị** `0 … m·n−1`. Vì giá trị phân biệt, mỗi ô ứng với đúng một hàng của `moveCost`. Cột đích `j` là chỉ số cột (0 … n−1), **không phải** giá trị ô đích.

In [3]:
g, mc, _ = examples[0]
print("ô (0,0) có giá trị", g[0][0], "-> chi phí sang cột 0/1 của hàng 1:", mc[g[0][0]])
print("nhầm dùng chỉ số cột 0 làm hàng của moveCost:", mc[0], "(sai)")

ô (0,0) có giá trị 5 -> chi phí sang cột 0/1 của hàng 1: [14, 3]
nhầm dùng chỉ số cột 0 làm hàng của moveCost: [9, 8] (sai)


### 3.3 Nguyên lý tối ưu con (optimal substructure)

Nếu đường tối ưu tới `(i, j)` đi qua `(i − 1, k)` thì đoạn đầu của nó phải là đường tối ưu tới `(i − 1, k)` — nếu không, thay đoạn đầu bằng đường rẻ hơn sẽ được đường rẻ hơn tới `(i, j)`. Đây là cơ sở của DP theo hàng.

### 3.4 Đồ thị phân tầng là DAG

Cạnh chỉ đi xuống ⇒ thứ tự hàng là thứ tự tô-pô ⇒ DP theo hàng chính là "đường đi ngắn nhất trên DAG".

## 4. Ý tưởng nền tảng / trực giác

Đặt `f[i][j]` = chi phí nhỏ nhất của một đường đi bắt đầu ở hàng 0 và **kết thúc tại** `(i, j)` (đã tính cả giá trị ô `(i, j)`):

- `f[0][j] = grid[0][j]`;
- `f[i][j] = grid[i][j] + min_k ( f[i−1][k] + moveCost[grid[i−1][k]][j] )`;
- đáp án `= min_j f[m−1][j]`.

Tính tay ví dụ 1 (`grid = [[5,3],[4,0],[2,1]]`):

| hàng | cột 0 | cột 1 |
|---|---|---|
| 0 | 5 | 3 |
| 1 | 4 + min(5 + mc[5][0]=14, 3 + mc[3][0]=18) = 4 + 19 = **23** | 0 + min(5 + mc[5][1]=3, 3 + mc[3][1]=6) = 0 + 8 = **8** |
| 2 | 2 + min(23 + mc[4][0]=2, 8 + mc[0][0]=9) = 2 + 17 = **19** | 1 + min(23 + mc[4][1]=4, 8 + mc[0][1]=8) = 1 + 16 = **17** |

`min(19, 17) = 17` ✔.

In [4]:
g, mc, _ = examples[0]
f = g[0][:]
print("hàng 0:", f)
for i in range(1, len(g)):
    new = []
    for j in range(len(g[0])):
        cands = [f[k] + mc[g[i - 1][k]][j] for k in range(len(g[0]))]
        new.append(g[i][j] + min(cands))
        print(f"  ({i},{j}): giá trị {g[i][j]} + min{cands} = {new[-1]}")
    f = new; print("hàng", i, ":", f)
print("đáp án:", min(f))

hàng 0: [5, 3]
  (1,0): giá trị 4 + min[19, 21] = 23
  (1,1): giá trị 0 + min[8, 9] = 8
hàng 1 : [23, 8]
  (2,0): giá trị 2 + min[25, 17] = 19
  (2,1): giá trị 1 + min[27, 16] = 17
hàng 2 : [19, 17]
đáp án: 17


## 5. Cách 1 — Brute Force / baseline

### 5.1 Duyệt mọi dãy cột — O(nᵐ · m)

In [5]:
def brute(grid, moveCost):
    m, n = len(grid), len(grid[0])
    return min(path_cost(grid, moveCost, cols) for cols in product(range(n), repeat=m))

for g, mc, exp in examples:
    print(brute(g, mc), "OK" if brute(g, mc) == exp else "SAI")

17 OK
6 OK


Với `m = n = 50`: `50⁵⁰` đường — bất khả thi. Chỉ dùng kiểm chứng (`nᵐ ≤ ~5·10⁴`).

### 5.2 Baseline 2 — Đệ quy có nhớ từ trên xuống — O(m · n²)

`best(i, j)` = chi phí nhỏ nhất **từ** `(i, j)` (tính cả giá trị ô này) tới hàng cuối.

In [6]:
def memo(grid, moveCost):
    m, n = len(grid), len(grid[0])
    @lru_cache(maxsize=None)
    def best(i, j):
        v = grid[i][j]
        if i == m - 1: return v
        return v + min(moveCost[v][k] + best(i + 1, k) for k in range(n))
    r = min(best(0, j) for j in range(n)); best.cache_clear()
    return r

for g, mc, exp in examples:
    print(memo(g, mc), "OK" if memo(g, mc) == exp else "SAI")

17 OK
6 OK


Đúng và đủ nhanh (`50 · 50² = 1,25·10⁵`), nhưng dùng đệ quy (độ sâu `m ≤ 50`, an toàn) và bộ nhớ đệm O(m·n). **Nút thắt** của vét cạn là các đường đi chia sẻ phần đuôi/đầu; DP gộp chúng theo trạng thái `(hàng, cột)`.

## 6. Ý tưởng thuật toán / bước đột phá

**DP lặp theo hàng, giữ hai hàng.**

```
f = grid[0][:]
for i in 1..m−1:
    g[j] = grid[i][j] + min_k (f[k] + moveCost[grid[i−1][k]][j])
    f = g
return min(f)
```

- Số trạng thái `m·n`, mỗi trạng thái xét `n` tiền nhiệm ⇒ **O(m · n²)** ≈ `1,25·10⁵` phép tính.
- Bộ nhớ O(n).

**Mẹo cài đặt nhanh trong Python:** với mỗi ô nguồn `k` của hàng trước, hàng chi phí `moveCost[grid[i−1][k]]` là một danh sách độ dài `n`; cộng nó với `f[k]` rồi lấy `min` theo cột bằng `zip(*...)` hoặc `map(min, ...)`.

### 6.1 Vì sao không thể "chọn ô nhỏ nhất mỗi hàng"?

Chi phí bước **phụ thuộc vào cặp** (ô nguồn, cột đích) — một ô giá trị nhỏ có thể có `moveCost` rất lớn tới mọi cột của hàng sau. Vì thế các hàng **không độc lập**; chỉ khi mọi `moveCost` bằng nhau (mục 12.3) thì đáp án mới bằng tổng min mỗi hàng cộng `(m − 1)·c`. DP xử lý đúng sự phụ thuộc này bằng cách giữ **mọi** cột của hàng trước làm trạng thái.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DP lặp theo hàng (cách doocs)

In [7]:
def dp_rows(grid, moveCost):
    m, n = len(grid), len(grid[0])
    f = grid[0]
    for i in range(1, m):
        g = [inf] * n
        for j in range(n):
            for k in range(n):
                g[j] = min(g[j], f[k] + moveCost[grid[i - 1][k]][j] + grid[i][j])
        f = g
    return min(f)

for g, mc, exp in examples:
    print(dp_rows(g, mc), "OK" if dp_rows(g, mc) == exp else "SAI")

17 OK
6 OK


### 7.2 Cách 3 — DP "đẩy" (push) từ ô nguồn, gộp bằng `map(min, ...)`

In [8]:
def dp_push(grid, moveCost):
    f = grid[0][:]
    for i in range(1, len(grid)):
        rows = [[fk + c for c in moveCost[v]] for fk, v in zip(f, grid[i - 1])]   # rows[k][j]
        f = [x + y for x, y in zip(map(min, zip(*rows)), grid[i])]
    return min(f)

for g, mc, exp in examples:
    print(dp_push(g, mc), "OK" if dp_push(g, mc) == exp else "SAI")

17 OK
6 OK


`zip(*rows)` chuyển vị để mỗi phần tử là cột `j` (các ứng viên từ mọi `k`); `map(min, …)` lấy min từng cột. Ít vòng lặp Python hơn ⇒ nhanh hơn vài lần.

### 7.3 Cách 4 — DP từ dưới lên (chi phí "từ ô này tới đích")

In [9]:
def dp_bottom_up(grid, moveCost):
    m, n = len(grid), len(grid[0])
    h = grid[m - 1][:]                              # h[j]: chi phí nhỏ nhất từ (i, j) tới hàng cuối
    for i in range(m - 2, -1, -1):
        h = [grid[i][j] + min(moveCost[grid[i][j]][k] + h[k] for k in range(n)) for j in range(n)]
    return min(h)

for g, mc, exp in examples:
    print(dp_bottom_up(g, mc), "OK" if dp_bottom_up(g, mc) == exp else "SAI")

17 OK
6 OK


Hướng ngược tự nhiên hơn với cách đánh chỉ số `moveCost[giá trị ô hiện tại][cột kế]`: ô hiện tại là nguồn, nên đọc ngay hàng `moveCost` của nó.

### 7.4 Cách 5 — Dijkstra trên đồ thị `m·n + 2` đỉnh (tổng quát, không tận dụng DAG)

Thêm đỉnh nguồn ảo nối tới hàng 0 và đỉnh đích ảo từ hàng cuối. Trọng số đỉnh (giá trị ô) chuyển thành trọng số cạnh đi **vào** ô.

In [10]:
def dijkstra(grid, moveCost):
    m, n = len(grid), len(grid[0])
    dist = {}; pq = [(grid[0][j], 0, j) for j in range(n)]; heapq.heapify(pq)
    while pq:
        d, i, j = heapq.heappop(pq)
        if (i, j) in dist: continue
        dist[(i, j)] = d
        if i == m - 1: return d                     # lần đầu chạm hàng cuối là tối ưu (trọng số dương)
        v = grid[i][j]
        for k in range(n):
            if (i + 1, k) not in dist:
                heapq.heappush(pq, (d + moveCost[v][k] + grid[i + 1][k], i + 1, k))

for g, mc, exp in examples:
    print(dijkstra(g, mc), "OK" if dijkstra(g, mc) == exp else "SAI")

17 OK
6 OK


Đúng vì mọi trọng số ≥ 0, nhưng O(m·n² log(m·n)) — chậm hơn DP và không cần thiết.

## 8. TRACE từng bước

### 8.1 Trace DP theo hàng trên ví dụ 2 (in đủ mọi ứng viên)

In [11]:
def trace(grid, moveCost):
    m, n = len(grid), len(grid[0]); f = grid[0][:]
    print("f hàng 0 =", f)
    for i in range(1, m):
        g = []
        for j in range(n):
            terms = [f"{f[k]}+mc[{grid[i-1][k]}][{j}]={f[k] + moveCost[grid[i-1][k]][j]}" for k in range(n)]
            g.append(grid[i][j] + min(f[k] + moveCost[grid[i - 1][k]][j] for k in range(n)))
            print(f"  ô ({i},{j}) giá trị {grid[i][j]}: min({', '.join(terms)}) + {grid[i][j]} = {g[-1]}")
        f = g; print(f"f hàng {i} =", f)
    return min(f)

print("đáp án:", trace(*examples[1][:2]))

f hàng 0 = [5, 1, 2]
  ô (1,0) giá trị 4: min(5+mc[5][0]=10, 1+mc[1][0]=21, 2+mc[2][0]=23) + 4 = 14
  ô (1,1) giá trị 0: min(5+mc[5][1]=8, 1+mc[1][1]=24, 2+mc[2][1]=9) + 0 = 8
  ô (1,2) giá trị 3: min(5+mc[5][2]=7, 1+mc[1][2]=9, 2+mc[2][2]=3) + 3 = 6
f hàng 1 = [14, 8, 6]
đáp án: 6


Ô `(1, 2)` (giá trị 3): ứng viên tốt nhất đến từ ô giá trị 2 (cột 2 hàng 0): `2 + moveCost[2][2] = 2 + 1 = 3`, cộng giá trị 3 ⇒ 6.

### 8.2 Truy vết đường đi tối ưu

In [12]:
def best_path(grid, moveCost):
    m, n = len(grid), len(grid[0]); f = grid[0][:]; par = [[-1] * n for _ in range(m)]
    for i in range(1, m):
        g = [0] * n
        for j in range(n):
            k = min(range(n), key=lambda k: f[k] + moveCost[grid[i - 1][k]][j])
            par[i][j] = k; g[j] = grid[i][j] + f[k] + moveCost[grid[i - 1][k]][j]
        f = g
    j = min(range(n), key=f.__getitem__); cols = [j]
    for i in range(m - 1, 0, -1):
        j = par[i][j]; cols.append(j)
    cols.reverse()
    return min(f), cols, [grid[i][c] for i, c in enumerate(cols)]

for g, mc, exp in examples:
    cost, cols, vals = best_path(g, mc)
    assert path_cost(g, mc, cols) == cost
    print(f"chi phí {cost}, cột {cols}, giá trị trên đường {vals}")

chi phí 17, cột [0, 1, 1], giá trị trên đường [5, 0, 1]
chi phí 6, cột [2, 2], giá trị trên đường [2, 3]


## 9. Code nộp LeetCode

In [13]:
from typing import List

class Solution:
    def minPathCost(self, grid: List[List[int]], moveCost: List[List[int]]) -> int:
        n = len(grid[0])
        f = grid[0][:]                                  # f[j]: chi phí nhỏ nhất kết thúc ở (i, j)
        for i in range(1, len(grid)):
            prev_vals = grid[i - 1]
            g = []
            for j in range(n):
                best = min(f[k] + moveCost[prev_vals[k]][j] for k in range(n))   # moveCost[GIÁ TRỊ ô nguồn][cột đích]
                g.append(best + grid[i][j])
            f = g
        return min(f)

In [14]:
sol = Solution()
for g, mc, exp in examples:
    got = sol.minPathCost(g, mc)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

17 | kỳ vọng 17 | OK
6 | kỳ vọng 6 | OK


## 10. Vì sao thuật toán đúng

**Định nghĩa.** `F(i, j)` = min chi phí trên các dãy cột `c₀ … cᵢ` với `cᵢ = j`, chi phí tính theo công thức mục 3.1 giới hạn ở `i + 1` hàng đầu.

**Truy hồi.** Với `i ≥ 1`, mọi dãy như vậy có dạng (dãy `c₀ … c_{i−1}` kết thúc ở `k`) + bước `k → j` + ô `(i, j)`:

`cost = cost(c₀ … c_{i−1}) + moveCost[grid[i−1][k]][j] + grid[i][j]`.

Với `k` cố định, phần đầu nhỏ nhất là `F(i−1, k)` và hai số hạng còn lại không phụ thuộc phần đầu ⇒ `F(i, j) = grid[i][j] + min_k (F(i−1, k) + moveCost[grid[i−1][k]][j])`. Cơ sở `F(0, j) = grid[0][j]`.

**Đáp án.** Đường đi phải kết thúc ở hàng `m − 1` tại một cột nào đó ⇒ `min_j F(m−1, j)`.

**Nén.** `F(i, ·)` chỉ phụ thuộc `F(i−1, ·)` ⇒ giữ một hàng `f` và tạo hàng mới `g` (không ghi đè `f` trong lúc tính `g` vì mọi `g[j]` cần **toàn bộ** `f`). ∎

**Ghi chú:** khác với nhiều bài DP lưới (chỉ đi xuống/chéo), ở đây mỗi ô có `n` tiền nhiệm, nên không thể cập nhật tại chỗ trên một mảng duy nhất.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mọi dãy cột | O(nᵐ · m) | O(m) |
| 1b. Đệ quy có nhớ | O(m · n²) | O(m · n) |
| 2. DP theo hàng | O(m · n²) | O(n) |
| 3. DP đẩy + `map(min)` | O(m · n²) | O(n²) tạm |
| 4. DP từ dưới lên | O(m · n²) | O(n) |
| 5. Dijkstra | O(m · n² · log(mn)) | O(m · n²) heap |

In [15]:
rng = random.Random(2304)
m = n = 50
vals = list(range(m * n)); rng.shuffle(vals)
G = [vals[i * n:(i + 1) * n] for i in range(m)]
MC = [[rng.randint(1, 100) for _ in range(n)] for _ in range(m * n)]
res = []
for name, f in [("memo", memo), ("dp_rows", dp_rows), ("dp_push", dp_push), ("dp_bottom_up", dp_bottom_up),
                ("dijkstra", dijkstra), ("Solution", Solution().minPathCost)]:
    t0 = time.perf_counter(); res.append(f(G, MC)); dt = time.perf_counter() - t0
    print(f"{name:<13} {dt*1000:7.1f} ms -> {res[-1]}")
assert len(set(res)) == 1

memo             14.9 ms -> 4487
dp_rows          12.3 ms -> 4487
dp_push           5.0 ms -> 4487
dp_bottom_up      6.8 ms -> 4487
dijkstra         40.7 ms -> 4487


Solution          7.4 ms -> 4487


## 12. Bộ test

### 12.1 Ví dụ đề + edge case (đối chiếu vét cạn)

In [16]:
def make_case(rng, m, n, lo=1, hi=100):
    vals = list(range(m * n)); rng.shuffle(vals)
    grid = [vals[i * n:(i + 1) * n] for i in range(m)]
    mc = [[rng.randint(lo, hi) for _ in range(n)] for _ in range(m * n)]
    return grid, mc

ALL = {"brute": brute, "memo": memo, "dp_rows": dp_rows, "dp_push": dp_push, "dp_bottom_up": dp_bottom_up,
       "dijkstra": dijkstra, "Solution": Solution().minPathCost}
edge = examples + [
    ([[0, 1], [2, 3]], [[1, 1], [1, 1], [1, 1], [1, 1]], 3),             # 0 -> 2: 0 + 2 + 1
    ([[3, 2], [1, 0]], [[100, 100], [100, 100], [100, 1], [1, 100]], 3),   # đường tốt nhất 2 -> 0 (mc[2][1] = 1)
    ([[0, 1, 2], [3, 4, 5]], [[1] * 3 for _ in range(6)], 4),             # chi phí bước đều = 1: 0 + 1 + 3
]
for g, mc, exp in edge:
    ref = brute(g, mc)
    for name, f in ALL.items():
        assert f(g, mc) == ref, (name, g, mc)
    print(f"grid={g!s:<28} -> {ref} | ghi trong bảng: {exp} | khớp: {ref == exp}")

grid=[[5, 3], [4, 0], [2, 1]]     -> 17 | ghi trong bảng: 17 | khớp: True
grid=[[5, 1, 2], [4, 0, 3]]       -> 6 | ghi trong bảng: 6 | khớp: True
grid=[[0, 1], [2, 3]]             -> 3 | ghi trong bảng: 3 | khớp: True
grid=[[3, 2], [1, 0]]             -> 3 | ghi trong bảng: 3 | khớp: True
grid=[[0, 1, 2], [3, 4, 5]]       -> 4 | ghi trong bảng: 4 | khớp: True


Case `[[3, 2], [1, 0]]`: bốn đường `3→1`, `3→0`, `2→1`, `2→0` có chi phí `3 + mc[3][0] + 1 = 5`, `3 + mc[3][1] + 0 = 103`, `2 + mc[2][0] + 1 = 103`, `2 + mc[2][1] + 0 = 3` ⇒ đáp án **3** (đường `2 → 0`). Cột `khớp:` xác nhận kỳ vọng ghi tay.

### 12.2 Đối chiếu ngẫu nhiên với vét cạn (seed cố định)

In [17]:
rng = random.Random(42)
for _ in range(400):
    m, n = rng.randint(2, 5), rng.randint(2, 4)
    if n ** m > 3000: continue
    g, mc = make_case(rng, m, n, 1, rng.choice([3, 100]))
    ref = brute(g, mc)
    for name, f in ALL.items():
        assert f(g, mc) == ref, (name, g, mc)
print("đồ thị ngẫu nhiên OK")

đồ thị ngẫu nhiên OK


### 12.3 Tính chất: đáp án ≥ (min mỗi hàng cộng lại) + (m − 1)·min(moveCost); tăng mọi `moveCost` thêm `d` ⇒ đáp án tăng đúng `(m − 1)·d`

In [18]:
rng = random.Random(7)
for _ in range(300):
    m, n = rng.randint(2, 6), rng.randint(2, 6); g, mc = make_case(rng, m, n)
    ans = Solution().minPathCost(g, mc)
    lower = sum(min(r) for r in g) + (m - 1) * min(min(r) for r in mc)
    assert ans >= lower
    d = rng.randint(1, 20)
    assert Solution().minPathCost(g, [[x + d for x in r] for r in mc]) == ans + (m - 1) * d
print("tính chất OK")

tính chất OK


Tính chất thứ hai đúng vì **mọi** đường đi có đúng `m − 1` bước — cộng hằng số vào mọi bước không đổi đường tối ưu.

### 12.4 Edge cực trị: lưới "dài" 50 × 2 và lưới "rộng" 2 × 50

Lưới rộng 2 × 50 có `50² = 2500` đường — vét cạn được; lưới dài 50 × 2 có `2⁵⁰` đường — chỉ so các cách DP với nhau và với Dijkstra.

In [19]:
rng = random.Random(124)
g, mc = make_case(rng, 2, 50)
assert Solution().minPathCost(g, mc) == brute(g, mc) == dijkstra(g, mc)
g, mc = make_case(rng, 50, 2)
vals = {name: f(g, mc) for name, f in ALL.items() if name != "brute"}
assert len(set(vals.values())) == 1
print("2x50 khớp vét cạn; 50x2 mọi cách DP/Dijkstra cho", vals["Solution"])

2x50 khớp vét cạn; 50x2 mọi cách DP/Dijkstra cho 3681


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng **chỉ số cột** nguồn thay vì **giá trị** ô nguồn để tra `moveCost`

In [20]:
def wrong_index(grid, moveCost):
    n = len(grid[0]); f = grid[0][:]
    for i in range(1, len(grid)):
        f = [grid[i][j] + min(f[k] + moveCost[k][j] for k in range(n)) for j in range(n)]   # SAI: moveCost[k]
    return min(f)

for g, mc, _ in examples:
    print("bản sai :", wrong_index(g, mc), "| bản đúng:", Solution().minPathCost(g, mc))

bản sai : 11 | bản đúng: 17
bản sai : 6 | bản đúng: 6


**Vì sao sai:** `moveCost[k]` là chi phí của ô **có giá trị `k`**, không phải ô ở cột `k`. Ví dụ 2 **trùng hợp** đúng: đường tối ưu `2 → 3` có `moveCost[2][2] = 1`, và bản sai tra `moveCost[2][2]` cho ô ở cột 2 — đúng là ô giá trị 2 nằm ở cột 2. Ví dụ 1 thì ô giá trị 5 nằm ở cột 0 nên lộ lỗi. **Sửa:** `moveCost[grid[i−1][k]][j]`.

### Lỗi 2 — Quên cộng giá trị ô (chỉ cộng chi phí bước)

In [21]:
def wrong_no_cell(grid, moveCost):
    n = len(grid[0]); f = [0] * n                                  # SAI: bỏ giá trị ô
    for i in range(1, len(grid)):
        f = [min(f[k] + moveCost[grid[i - 1][k]][j] for k in range(n)) for j in range(n)]
    return min(f)

for g, mc, _ in examples:
    print("bản sai :", wrong_no_cell(g, mc), "| bản đúng:", Solution().minPathCost(g, mc))

bản sai : 11 | bản đúng: 17
bản sai : 1 | bản đúng: 6


**Sửa:** chi phí đường = tổng giá trị ô + tổng chi phí bước; khởi tạo `f = grid[0]` và cộng `grid[i][j]` mỗi hàng.

### Lỗi 3 — Tham lam: ở mỗi hàng đi tới ô có `giá trị + chi phí bước` nhỏ nhất

In [22]:
def wrong_greedy(grid, moveCost):
    m, n = len(grid), len(grid[0]); best = inf
    for start in range(n):
        j = start; total = grid[0][j]
        for i in range(1, m):
            v = grid[i - 1][j]
            j = min(range(n), key=lambda k: moveCost[v][k] + grid[i][k])     # SAI: chỉ nhìn một bước
            total += moveCost[v][j] + grid[i][j]
        best = min(best, total)
    return best

for g, mc, _ in examples:
    print("bản sai :", wrong_greedy(g, mc), "| bản đúng:", Solution().minPathCost(g, mc))
rng = random.Random(3); found = None
for _ in range(5000):
    g, mc = make_case(rng, 3, 3)
    if wrong_greedy(g, mc) != Solution().minPathCost(g, mc): found = (g, mc); break
print("phản ví dụ 3x3: bản sai", wrong_greedy(*found), "| bản đúng", Solution().minPathCost(*found))

bản sai : 17 | bản đúng: 17
bản sai : 6 | bản đúng: 6
phản ví dụ 3x3: bản sai 40 | bản đúng 22


**Vì sao sai:** cả hai ví dụ của đề **trùng hợp** đúng (ví dụ 2 chỉ có một bước, tham lam một bước là tối ưu). Một bước rẻ có thể dẫn tới ô có chi phí bước tiếp theo rất đắt. **Sửa:** DP xét mọi tiền nhiệm.

### Lỗi 4 — Cập nhật **tại chỗ** trên một mảng `f`

In [23]:
def wrong_inplace(grid, moveCost):
    n = len(grid[0]); f = grid[0][:]
    for i in range(1, len(grid)):
        for j in range(n):
            f[j] = grid[i][j] + min(f[k] + moveCost[grid[i - 1][k]][j] for k in range(n))   # SAI: f[k<j] đã là hàng mới
    return min(f)

for g, mc, _ in examples:
    print("bản sai :", wrong_inplace(g, mc), "| bản đúng:", Solution().minPathCost(g, mc))

bản sai : 18 | bản đúng: 17
bản sai : 6 | bản đúng: 6


**Vì sao sai:** khi tính `f[1]`, `f[0]` đã là giá trị của hàng **mới** — trộn hai hàng. Ví dụ 2 **trùng hợp** đúng: chỉ có một lần chuyển hàng, và ô tối ưu `(1, 2)` lấy giá trị tốt nhất từ `f[2]` — ô duy nhất **chưa** bị ghi đè tại thời điểm tính `f[2]`. **Sửa:** tạo mảng `g` mới cho mỗi hàng.

### Lỗi 5 — Cộng thêm chi phí bước **từ hàng cuối**

In [24]:
def wrong_last_move(grid, moveCost):
    m, n = len(grid), len(grid[0]); f = grid[0][:]
    for i in range(1, m):
        f = [grid[i][j] + min(f[k] + moveCost[grid[i - 1][k]][j] for k in range(n)) for j in range(n)]
    return min(f[j] + min(moveCost[grid[m - 1][j]]) for j in range(n))   # SAI: cộng thêm một bước "đi tiếp" từ hàng cuối

for g, mc, _ in examples:
    print("bản sai :", wrong_last_move(g, mc), "| bản đúng:", Solution().minPathCost(g, mc))

bản sai : 18 | bản đúng: 17
bản sai : 7 | bản đúng: 6


**Sửa:** đường đi kết thúc ở hàng cuối, không có bước nào sau đó.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đi từ hàng này sang hàng kế, chọn cột bất kỳ, tối thiểu tổng chi phí" ⇒ **DP theo tầng**, O(m · n²).
- Đọc kỹ chỉ số của bảng chi phí: theo **giá trị** hay theo **vị trí**.
- Mỗi ô có nhiều tiền nhiệm ⇒ **không** cập nhật tại chỗ; dùng hai hàng.
- Đồ thị phân tầng là DAG ⇒ không cần Dijkstra.
- Tham lam một bước gần như luôn sai ở DP đường đi.
- Bài cùng họ: 931 Minimum Falling Path Sum, 1289 Minimum Falling Path Sum II, 120 Triangle, 64 Minimum Path Sum, 1981 Minimize the Difference Between Target and Chosen Elements, 2435.

## 15. Biến thể của bài

### 15.1 Biến thể A — Chỉ được đi xuống cột **kề** (`j − 1, j, j + 1`) — giống LC 931

In [25]:
def adjacent_only(grid, moveCost):
    m, n = len(grid), len(grid[0]); f = grid[0][:]
    for i in range(1, m):
        f = [grid[i][j] + min(f[k] + moveCost[grid[i - 1][k]][j] for k in range(max(0, j - 1), min(n, j + 2)))
             for j in range(n)]
    return min(f)

def adjacent_brute(grid, moveCost):
    m, n = len(grid), len(grid[0])
    return min(path_cost(grid, moveCost, cols) for cols in product(range(n), repeat=m)
               if all(abs(a - b) <= 1 for a, b in zip(cols, cols[1:])))

rng = random.Random(151)
for _ in range(200):
    g, mc = make_case(rng, rng.randint(2, 4), rng.randint(2, 4))
    assert adjacent_only(g, mc) == adjacent_brute(g, mc)
print(adjacent_only(*examples[0][:2]), adjacent_only(*examples[1][:2]))

17 6


### 15.2 Biến thể B — Đếm số đường đi có chi phí nhỏ nhất

In [26]:
def count_optimal(grid, moveCost):
    m, n = len(grid), len(grid[0]); f = grid[0][:]; cnt = [1] * n
    for i in range(1, m):
        g, c = [], []
        for j in range(n):
            cand = [f[k] + moveCost[grid[i - 1][k]][j] for k in range(n)]
            b = min(cand); g.append(b + grid[i][j]); c.append(sum(cnt[k] for k in range(n) if cand[k] == b))
        f, cnt = g, c
    b = min(f)
    return sum(cnt[j] for j in range(n) if f[j] == b)

def count_optimal_brute(grid, moveCost):
    m, n = len(grid), len(grid[0])
    costs = [path_cost(grid, moveCost, cols) for cols in product(range(n), repeat=m)]
    return costs.count(min(costs))

rng = random.Random(152)
for _ in range(200):
    g, mc = make_case(rng, rng.randint(2, 4), rng.randint(2, 3), 1, 2)
    assert count_optimal(g, mc) == count_optimal_brute(g, mc)
print(count_optimal(*examples[0][:2]), count_optimal([[0, 1], [2, 3]], [[1, 1]] * 4))

1 1


### 15.3 Biến thể C — Chi phí **lớn nhất** (đổi `min` thành `max`)

Vì đồ thị là DAG, đường đi dài nhất cũng giải được bằng DP (khác với đồ thị tổng quát có chu trình).

In [27]:
def max_path_cost(grid, moveCost):
    n = len(grid[0]); f = grid[0][:]
    for i in range(1, len(grid)):
        f = [grid[i][j] + max(f[k] + moveCost[grid[i - 1][k]][j] for k in range(n)) for j in range(n)]
    return max(f)

rng = random.Random(153)
for _ in range(200):
    g, mc = make_case(rng, rng.randint(2, 4), rng.randint(2, 3))
    m, n = len(g), len(g[0])
    assert max_path_cost(g, mc) == max(path_cost(g, mc, cols) for cols in product(range(n), repeat=m))
print(max_path_cost(*examples[0][:2]))

30


### 15.4 Biến thể D — Bắt buộc bắt đầu ở cột `s` và kết thúc ở cột `t`

In [28]:
def fixed_ends(grid, moveCost, s, t):
    n = len(grid[0]); f = [inf] * n; f[s] = grid[0][s]
    for i in range(1, len(grid)):
        f = [grid[i][j] + min(f[k] + moveCost[grid[i - 1][k]][j] for k in range(n)) for j in range(n)]
    return f[t]

g, mc, _ = examples[0]
print({(s, t): fixed_ends(g, mc, s, t) for s in range(2) for t in range(2)})

{(0, 0): 19, (0, 1): 17, (1, 0): 20, (1, 1): 18}


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi dãy cột | định nghĩa | O(nᵐ·m) | O(m) | kiểm chứng |
| 1b. Đệ quy có nhớ | top-down | O(m·n²) | O(m·n) | dễ viết |
| 2. DP theo hàng | kéo (pull) | O(m·n²) | O(n) | **mặc định** |
| 3. DP đẩy + `map(min)` | vector hóa | O(m·n²) | O(n²) | nhanh trong Python |
| 4. DP từ dưới lên | chi phí còn lại | O(m·n²) | O(n) | chỉ số `moveCost` tự nhiên |
| 5. Dijkstra | đồ thị tổng quát | O(m·n² log) | O(m·n²) | khi không có cấu trúc tầng |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `grid = [[0, 1], [2, 3]]`, `moveCost = [[5, 1], [1, 5], [1, 1], [1, 1]]`?

**Bài 2 (tính tay):** `grid = [[2, 0], [1, 3], [4, 5]]`, mọi `moveCost[v] = [1, 2]`. Tính `f` từng hàng.

**Bài 3 (điền TODO):** Viết `bt3_todo(grid, moveCost)` trả về **danh sách giá trị ô** trên một đường đi tối ưu.

**Bài 4 (tự viết lại):** Viết `bt4_todo(grid, moveCost)` cài DP **từ dưới lên** không nhìn code.

**Bài 5 (mini challenge):** Viết `bt5_todo(grid, moveCost, k)` trả về chi phí nhỏ nhất khi được phép **miễn phí tối đa `k` bước** (chi phí bước đó = 0).

In [29]:
def bt3_todo(grid, moveCost):
    # TODO: giá trị ô trên đường tối ưu
    pass

def bt4_todo(grid, moveCost):
    # TODO: DP từ dưới lên
    pass

def bt5_todo(grid, moveCost, k):
    # TODO: miễn phí tối đa k bước
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: bốn đường. Mức 2: `0 → 3`: `0 + 1 + 3 = 4`; `0 → 2`: `0 + 5 + 2 = 7`; `1 → 2`: `1 + 1 + 2 = 4`; `1 → 3`: `1 + 5 + 3 = 9`. Mức 3: `4`.

**Bài 2.** Mức 1: `f₀ = [2, 0]`. Mức 2: `f₁ = [1 + min(2+1, 0+1), 3 + min(2+2, 0+2)] = [2, 5]`. Mức 3: `f₂ = [4 + min(2+1, 5+1), 5 + min(2+2, 5+2)] = [7, 9]` ⇒ `7`.

**Bài 3.** Mức 1: lưu `par[i][j]`. Mức 2: chọn cột cuối có `f` nhỏ nhất. Mức 3: lần ngược theo `par`, đổi cột thành giá trị.

**Bài 4.** Mức 1: `h = grid[m−1]`. Mức 2: `h[j] = grid[i][j] + min(moveCost[grid[i][j]][k] + h[k])`. Mức 3: duyệt `i` giảm dần, trả `min(h)`.

**Bài 5.** Mức 1: thêm chiều "số bước miễn phí đã dùng" `t ≤ k`. Mức 2: `f[t][j]` từ `f[t][k'] + cost` hoặc `f[t−1][k'] + 0`. Mức 3: đáp án `min` trên mọi `t ≤ k`.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `4`. **Bài 2:** `f = [2,0] → [2,5] → [7,9]`, đáp án `7`.

In [30]:
assert Solution().minPathCost([[0, 1], [2, 3]], [[5, 1], [1, 5], [1, 1], [1, 1]]) == 4
assert Solution().minPathCost([[2, 0], [1, 3], [4, 5]], [[1, 2]] * 6) == 7

def bt3(grid, moveCost):
    return best_path(grid, moveCost)[2]

def bt4(grid, moveCost):
    return dp_bottom_up(grid, moveCost)

def bt5(grid, moveCost, k):
    m, n = len(grid), len(grid[0])
    f = [[grid[0][j] for j in range(n)]] + [[inf] * n for _ in range(k)]     # f[t][j]
    for i in range(1, m):
        g = [[inf] * n for _ in range(k + 1)]
        for t in range(k + 1):
            for j in range(n):
                best = min(f[t][q] + moveCost[grid[i - 1][q]][j] for q in range(n))
                if t: best = min(best, min(f[t - 1]))
                g[t][j] = best + grid[i][j]
        f = g
    return min(min(r) for r in f)

def bt5_brute(grid, moveCost, k):
    m, n = len(grid), len(grid[0]); best = inf
    for cols in product(range(n), repeat=m):
        steps = sorted((moveCost[grid[i][cols[i]]][cols[i + 1]] for i in range(m - 1)), reverse=True)
        best = min(best, sum(grid[i][c] for i, c in enumerate(cols)) + sum(steps[k:]))
    return best

rng = random.Random(1993)
for _ in range(200):
    g, mc = make_case(rng, rng.randint(2, 4), rng.randint(2, 3))
    vals = bt3(g, mc)
    assert vals[0] in g[0] and vals[-1] in g[-1]
    assert bt4(g, mc) == brute(g, mc)
    k = rng.randint(0, 3)
    assert bt5(g, mc, k) == bt5_brute(g, mc, k)
print(bt3(*examples[0][:2]), bt5(*examples[0][:2], 1), bt5(*examples[0][:2], 2))
print("Đáp án bài tập: tất cả assert OK")

[5, 0, 1] 9 4
Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
f = grid[0][:]
for i in range(1, m):
    f = [grid[i][j] + min(f[k] + moveCost[grid[i-1][k]][j] for k in range(n))   # GIÁ TRỊ nguồn, CỘT đích
         for j in range(n)]
return min(f)
```

**Checklist:**
- [ ] `moveCost[giá trị ô nguồn][cột đích]`.
- [ ] Cộng giá trị ô ở **mọi** hàng (kể cả hàng 0).
- [ ] Hai mảng `f`/`g`, không cập nhật tại chỗ.
- [ ] Không có chi phí bước sau hàng cuối.